# PP-MAE — All 4 Options (Google Colab)
**Pathology-Prior Masked Autoencoder for Brain MRI Denoising**

| Round | PP-MAE Option | Baselines |
|-------|---------------|-----------|
| 1 | CNN U-Net | DnCNN, UNet-L1, Noise2Noise, REDNet |
| 2 | ViT MAE 2D | VanillaMAE, SparK-CNN |
| 3 | Full Pipeline | MultiTaskUNet, TransUNet, UNETR, SwinUNETR, SeqPipeline |
| 4 | Swin Transformer | SwinIR-lite, Uformer-lite |

> ⚠️ **First:** `Runtime → Change runtime type → T4 GPU`

---
## Step 1 — Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
else:
    print('⚠️  No GPU — Runtime → Change runtime type → T4 GPU')
print(f'PyTorch {torch.__version__}')

---
## Step 2 — Install packages

In [ ]:
%%capture
!pip install nibabel scikit-image matplotlib kagglehub

---
## Step 3 — Clone the PP-MAE repository

In [ ]:
import os, sys

REPO_DIR = '/content/AL-ML'
BRANCH   = 'claude/general-session-gviGa'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull origin {BRANCH} -q
    print('✅ Repo updated')
else:
    !git clone -q -b {BRANCH} https://github.com/abizbright1/AL-ML.git {REPO_DIR}
    print('✅ Repo cloned')

sys.path.insert(0, os.path.join(REPO_DIR, 'pp_mae'))
print('Branch:', BRANCH)

---
## Step 4 — Load BraTS 2021 data from Kaggle

**You need a Kaggle account and API key.**

### How to get your API key (one-time setup)
1. Go to **[kaggle.com](https://www.kaggle.com)** → sign in
2. Click your profile photo (top right) → **Settings**
3. Scroll to **API** section → click **"Create New Token"**
4. A file `kaggle.json` downloads — open it, it looks like:
   ```json
   {"username": "yourname", "key": "abc123..."}
   ```
5. Paste those two values below ↓

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PASTE YOUR KAGGLE CREDENTIALS HERE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KAGGLE_USERNAME = 'your_username'    # ← replace
KAGGLE_KEY      = 'your_api_key'     # ← replace
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Credentials saved')

In [ ]:
import kagglehub

# Download BraTS 2021 Task 1
print('Downloading BraTS 2021 from Kaggle...')
print('(First download ~15 GB — takes 5–10 min. Re-runs use cache instantly.)')
print()

path = kagglehub.dataset_download("dschettler8845/brats-2021-task1")

print()
print('Path to dataset files:', path)
print('Contents:', os.listdir(path)[:10])

In [ ]:
# Auto-find the folder that contains BraTS subject sub-directories
# (kagglehub sometimes nests files 1–2 levels deep)

def find_brats_root(base_path):
    """Walk into nested dirs until children contain .nii.gz files."""
    from collections import deque
    queue = deque([base_path])
    best  = (0, base_path)
    seen  = set()
    while queue:
        cur = queue.popleft()
        if cur in seen:
            continue
        seen.add(cur)
        try:
            children = sorted(os.listdir(cur))
        except PermissionError:
            continue
        # Count children that look like BraTS subject dirs (contain NIfTI)
        n_subj = sum(
            1 for c in children
            if os.path.isdir(os.path.join(cur, c)) and
               any(f.endswith('.nii.gz') or f.endswith('.nii')
                   for f in os.listdir(os.path.join(cur, c)))
        )
        if n_subj > best[0]:
            best = (n_subj, cur)
        if n_subj >= 5:
            break
        for c in children:
            child_path = os.path.join(cur, c)
            if os.path.isdir(child_path):
                queue.append(child_path)
    return best[1]

BRATS_ROOT = find_brats_root(path)

# Show what we found
subjects = sorted([
    d for d in os.listdir(BRATS_ROOT)
    if os.path.isdir(os.path.join(BRATS_ROOT, d))
])

print(f'✅ BRATS_ROOT : {BRATS_ROOT}')
print(f'   Subjects   : {len(subjects)}')
if subjects:
    sample_dir = os.path.join(BRATS_ROOT, subjects[0])
    print(f'   Sample     : {subjects[0]}')
    print(f'   Files      : {sorted(os.listdir(sample_dir))}')

---
## Step 4B — Alternative: Use Google Drive zip instead of Kaggle
*Skip this if Kaggle worked above.*

In [ ]:
# Only run this cell if the Kaggle download above failed.
# Upload brats_datat.zip to Google Drive first, then set USE_DRIVE = True.

USE_DRIVE = False   # ← set True if using Drive instead of Kaggle

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ZIP = '/content/drive/MyDrive/brats_datat.zip'  # ← your Drive path
    !mkdir -p /content/BraTS2021_data
    !unzip -q {DRIVE_ZIP} -d /content/BraTS2021_data
    !for f in /content/BraTS2021_data/*.tar; do \
        tar -xf "$f" -C /content/BraTS2021_data/ 2>/dev/null || true; done
    BRATS_ROOT = find_brats_root('/content/BraTS2021_data')
    print(f'✅ BRATS_ROOT: {BRATS_ROOT}')

---
## Step 5 — Configure experiment settings

In [ ]:
# ── Edit these settings ───────────────────────────────────────────
ROUNDS       = '3,5'    # which rounds: 1=CNN 2=ViT 3=Pipeline 4=Swin 5=SOTA-2026
MAX_SUBJECTS = 30       # subjects to use (BraTS 2021 has 1251 total)
EPOCHS       = 20       # epochs per model (10=quick, 30=full)
SEG_EPOCHS   = 20       # segmentor pre-training epochs
PATCH_SIZE   = 96       # crop size — must be divisible by 8
SIGMA        = 0.08     # Rician noise sigma
# ─────────────────────────────────────────────────────────────────

# Save results to Drive if available, else local
OUT_DIR = ('/content/drive/MyDrive/PP_MAE_Results'
           if os.path.exists('/content/drive/MyDrive')
           else '/content/PP_MAE_Results')
os.makedirs(OUT_DIR, exist_ok=True)

import torch
print('Settings')
print(f'  Rounds       : {ROUNDS}')
print(f'  Subjects     : {MAX_SUBJECTS}  (of {len(subjects)} available)')
print(f'  Epochs       : {EPOCHS}')
print(f'  Device       : {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'  BraTS root   : {BRATS_ROOT}')
print(f'  Output dir   : {OUT_DIR}')

---
## Step 6 — Run the experiments

Trains every model and prints live progress. Estimated time on T4 GPU:

| Rounds | Subjects | Epochs | Approx time |
|--------|----------|--------|-------------|
| 3,5    | 30       | 20     | ~2.5 h      |
| 2,3,4  | 30       | 20     | ~3.0 h      |
| 1,2,3,4,5| 30   | 20     | ~4.5 h      |
| 2,3,4  | 50       | 30     | ~5 h        |

In [ ]:
import subprocess

cmd = [
    sys.executable,
    f'{REPO_DIR}/run_all_options.py',
    BRATS_ROOT,                        # real BraTS data path
    '--epochs',       str(EPOCHS),
    '--seg_epochs',   str(SEG_EPOCHS),
    '--patch_size',   str(PATCH_SIZE),
    '--sigma',        str(SIGMA),
    '--max_subjects', str(MAX_SUBJECTS),
    '--rounds',       ROUNDS,
    '--out',          OUT_DIR,
]

print('Command:', ' '.join(cmd))
print('─' * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print('\n✅ All experiments complete!')
else:
    print(f'\n❌ Exited with code {proc.returncode}')

---
## Step 7 — View results

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image

# Show results table
for csv_path in [os.path.join(OUT_DIR, 'options_results.csv'),
                 os.path.join(REPO_DIR, 'options_results.csv')]:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print('=== RESULTS TABLE ===')
        display(df.style.highlight_max(
            subset=df.select_dtypes('number').columns,
            color='#d4edda'
        ))
        break

# Show all plots
all_plots = sorted(set(
    glob.glob(os.path.join(REPO_DIR, 'options_*.png')) +
    glob.glob(os.path.join(OUT_DIR,  'options_*.png'))
))
for f in all_plots:
    print(f'\n▶  {os.path.basename(f)}')
    display(Image(f))

---
## Step 8 — Head-to-head comparison
*Run this after all rounds complete (Rounds 1–5 all supported).*

In [ ]:
for csv_path in [os.path.join(REPO_DIR, 'options_results.csv'),
                 os.path.join(OUT_DIR,  'options_results.csv')]:
    if os.path.exists(csv_path):
        r = subprocess.run(
            [sys.executable, f'{REPO_DIR}/compare_options.py', '--csv', csv_path],
            capture_output=True, text=True
        )
        print(r.stdout)
        for f in sorted(glob.glob(os.path.join(REPO_DIR, 'options_compare_*.png'))):
            print(f'\n▶  {os.path.basename(f)}')
            display(Image(f))
        break
else:
    print('Run all 4 rounds first  (set ROUNDS="1,2,3,4" in Step 5)')

---
## Step 9 — Grading pipeline

In [ ]:
cmd_g = [
    sys.executable,
    f'{REPO_DIR}/run_grading.py',
    BRATS_ROOT,
    '--max_subjects', str(MAX_SUBJECTS),
    '--out', OUT_DIR,
]
print('Running grading pipeline...')
proc_g = subprocess.Popen(
    cmd_g,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in proc_g.stdout:
    print(line, end='', flush=True)
proc_g.wait()

for f in sorted(glob.glob(os.path.join(REPO_DIR, 'grading_*.png'))):
    print(f'\n▶  {os.path.basename(f)}')
    display(Image(f))

---
## Step 10 — Download all results to your computer

In [ ]:
import zipfile
from google.colab import files as colab_files

output_files = sorted(set(
    glob.glob(os.path.join(REPO_DIR, 'options_*.png')) +
    glob.glob(os.path.join(REPO_DIR, 'options_*.csv')) +
    glob.glob(os.path.join(REPO_DIR, 'grading_*.png')) +
    glob.glob(os.path.join(REPO_DIR, 'grading_*.csv')) +
    glob.glob(os.path.join(OUT_DIR,  '*.png')) +
    glob.glob(os.path.join(OUT_DIR,  '*.csv'))
))

if output_files:
    zip_path = '/content/PP_MAE_results.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for f in output_files:
            zf.write(f, os.path.basename(f))
    print(f'Packed {len(output_files)} files → PP_MAE_results.zip')
    colab_files.download(zip_path)
else:
    print('No output files yet — run Steps 6–9 first.')

---
## Run individual rounds (optional)
*Use these if you want to run one round at a time.*

In [ ]:
# ── Round 1: CNN  (PP-MAE CNN vs DnCNN / UNet / Noise2Noise / REDNet)
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', BRATS_ROOT,
       '--rounds','1','--epochs',str(EPOCHS),'--max_subjects',str(MAX_SUBJECTS),'--out',OUT_DIR]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# ── Round 2: ViT  (ViT PP-MAE 2D vs VanillaMAE / SparK-CNN)
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', BRATS_ROOT,
       '--rounds','2','--epochs',str(EPOCHS),'--max_subjects',str(MAX_SUBJECTS),'--out',OUT_DIR]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# ── Round 3: Pipeline  (PP-MAE Pipeline vs MultiTaskUNet / UNETR / SwinUNETR / SeqPipeline)
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', BRATS_ROOT,
       '--rounds','3','--epochs',str(EPOCHS),'--max_subjects',str(MAX_SUBJECTS),'--out',OUT_DIR]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# ── Round 4: Swin  (Swin PP-MAE vs SwinIR-lite / Uformer-lite)
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', BRATS_ROOT,
       '--rounds','4','--epochs',str(EPOCHS),'--max_subjects',str(MAX_SUBJECTS),'--out',OUT_DIR]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

In [ ]:
# ── Round 5: SOTA 2021-2026  (PP-MAE Pipeline vs nnU-Net / TransBTS / MedSegDiff / SwinUNETR-v2 / MedSAM / MedNeXt)
# This is the primary PhD comparison — PP-MAE vs the best published methods from the last 5 years.
cmd = [sys.executable, f'{REPO_DIR}/run_all_options.py', BRATS_ROOT,
       '--rounds','5','--epochs',str(EPOCHS),'--max_subjects',str(MAX_SUBJECTS),'--out',OUT_DIR]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\nRound 5 complete — SOTA 2021-2026 comparison done!')

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `401 Unauthorized` on Kaggle download | Wrong username/key — re-check `kaggle.json` |
| `403 host_not_allowed` on Kaggle download | Kaggle blocks some Colab IPs — use **Step 4B** (Google Drive) instead |
| `No subject directories found` | Run the `find_brats_root` cell again, print `BRATS_ROOT` |
| `Missing modalities` warning | Check the sample files printed in Step 4 |
| `CUDA out of memory` | Lower `MAX_SUBJECTS=15` or `PATCH_SIZE=64` |
| `ModuleNotFoundError` | Re-run Step 3 (repo clone) |
| Session disconnects | Mount Drive — results save there as each round finishes |